# Repair timings and durability model

This notebook is set up for the measurements, but I haven't collected repair timings or written the probability model yet. The work left is in `README.md` here and `../docs/HANDS_ON.md`.


## Load the measurements

Run this from the repo root or this folder. Keep failed runs visible when checking the data. A repair request returning successfully doesn't necessarily mean the repair has finished.


In [ ]:
from pathlib import Path
import csv
import sys
import subprocess

ROOT = Path.cwd() if (Path.cwd() / "pom.xml").exists() else Path.cwd().parent
measurements = ROOT / "measurements/runs.csv"
if measurements.exists():
    with measurements.open(newline="") as source:
        runs = list(csv.DictReader(source))
else:
    runs = []
    print("No observations yet. Run scripts/measure.py around actual commands.")
runs[:10]


## Look at completed repairs

I'll only label a command `repair` if it waits for reconstruction, verification and the metadata update to finish. Keep repeated runs, byte-size definitions, machine settings and failures with the results.


In [ ]:
repair_runs = [row for row in runs if row["label"] == "repair"]
successful_repairs = [row for row in repair_runs if int(row["exit_code"]) == 0]
failed_repairs = [row for row in repair_runs if int(row["exit_code"]) != 0]
print(f"Completed: {len(successful_repairs)}, failed attempts: {len(failed_repairs)}")
successful_repairs[:10]


## Pick failure-rate assumptions

I plan to use [Backblaze Drive Stats](https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data), with the exact drive models, date range and download cited. Follow its usage terms. I need to explain drive-days, AFR versus annual probability, and why disk failure rates aren't this cluster's measured node failure rates. There's no need to download or commit the whole dataset by default.

The model also needs clear assumptions about correlated failures, hazard rates, spare nodes, repair queues and chunk loss versus file loss. Compare 3+2, 4+2 and three replicas using the same assumptions.


In [ ]:
# HANDS_ON: choose and justify failure-rate inputs from a cited dataset.
# No default failure rate or measured repair duration is supplied.
failure_assumptions = None


## Write and check the model

I still need to choose an analytical model or simulation, explain it and check simple cases and convergence. The function below is a placeholder. Its inputs can change as the design takes shape.


In [ ]:
def annual_chunk_loss_probability(*, failure_assumptions, repair_observations, data_shards, parity_shards):
    raise NotImplementedError("HANDS_ON: write and check the model before exporting probabilities")


## Plot the model results

Once it's working, export `results/model.csv` with `scheme,repair_seconds,annual_loss_probability`. For the parity plot, use `results/parity.csv` with `scheme,parity_shards,annual_loss_probability`. Keep model inputs, uncertainty and limitations next to the output. There are no example probabilities filled in.


In [ ]:
model_csv = ROOT / "durability/results/model.csv"
if model_csv.exists():
    subprocess.run([sys.executable, str(ROOT / "durability/plot.py"), "durability",
        str(model_csv), "--output", str(ROOT / "durability/results/repair-risk.png")], check=True)
else:
    print("No model results yet; implement and validate the model first.")


## What the results actually show

Record the setup and repeat counts, and keep measurements separate from predictions. Explain sensitivity, correlated failures, spare capacity and bandwidth limits. Backblaze data doesn't establish laptop prediction performance. With six nodes, two permanent losses also leave too few live nodes to restore five separate holders without replacement/returning capacity.
